In [1]:
# 05_policy_appendix.ipynb
# Appendix: national digital-education policy indicators (IEA ICILS 2023 National
# Contexts Survey) joined with cross-national results (reads cross_country.csv).
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, pyreadstat
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
# --- grayscale publication style ---
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"axes.edgecolor":"0.15","grid.color":"0.85","font.size":10,
                     "axes.titlesize":1,"figure.autolayout":True})
GRAY = ["0.20","0.45","0.65","0.80"]
# --- resolve folders relative to repo (no path is printed, for anonymity) ---
BASE = Path("..") if (Path("../data")).exists() else Path(".")
DATA, FIG, TAB = BASE/"data", BASE/"results/figures", BASE/"results/tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def savefig(fig, name):
    # save both formats at 600 dpi, no caption/title; paths are never printed
    for ext in ("png","pdf"): fig.savefig(FIG/(name+"."+ext), dpi=600, bbox_inches="tight")
    plt.close(fig)
PVS=[f"PV{i}CIL" for i in range(1,6)]
def wR2(y,yh,w):
    yb=np.average(y,weights=w); return 1-np.sum(w*(y-yh)**2)/np.sum(w*(y-yb)**2)

ncs,m=pyreadstat.read_sav(str(DATA/"ICILS2023NCS.sav"))
WANT={"Austria":"AUT","Germany":"DEU","Denmark":"DNK","Finland":"FIN","Italy":"ITA","Portugal":"PRT","France":"FRA","Korea":"KOR"}
def iso(n):
    for k,v in WANT.items():
        if k.lower() in str(n).lower(): return v
ncs["cc"]=ncs.Country.map(iso); sub=ncs[ncs.cc.notna()].drop_duplicates("cc").copy()
MAP={"NC3A06AA":{1:"Compulsory",2:"Optional",3:"Not taught"},
 "NC3B16A":{1:"Nat'l compulsory",2:"Nat'l sample",3:"Non-compulsory",4:"School-level",5:"None"},
 "NC3B10A":{1:"Explicit",2:"Implied",3:"No emphasis"},"NC3B11B":{1:"Explicit",2:"Implied",3:"No emphasis"},
 "NC3B18B1":{1:"No",2:"Yes"},"NC3B09G":{1:"Explicit",2:"Implied",3:"No emphasis"},
 "NC3B09C":{1:"Explicit",2:"Implied",3:"No emphasis"}}
NAMES={"NC3A06AA":"CIL_subject_ISCED2","NC3B16A":"CIL_assessment","NC3B10A":"ICT_provision_plan",
 "NC3B11B":"Inservice_ICT_plan","NC3B18B1":"Preservice_ICT_mandatory","NC3B09G":"Cyber_safety","NC3B09C":"Info_literacy"}
out=pd.DataFrame({"cc":sub.cc.values})
for c,mp in MAP.items(): out[NAMES[c]]=[mp.get(int(v),"-") if not pd.isna(v) else "-" for v in sub[c].values]
out["Compulsory_years"]=[int(v) if not pd.isna(v) and v<90 else None for v in sub["NC3A02B"].values]
cc=pd.read_csv(TAB/"cross_country.csv")[["cc","immig_pct","corr_base_sens","R2_res","R2_part"]]
out=out.merge(cc,on="cc",how="left").sort_values("cc").reset_index(drop=True)
out.to_csv(TAB/"policy_appendix.csv",index=False)
with open(TAB/"policy_appendix.tex","w") as f: f.write(out.to_latex(index=False))
print(out.to_string(index=False))


 cc CIL_subject_ISCED2   CIL_assessment ICT_provision_plan Inservice_ICT_plan Preservice_ICT_mandatory Cyber_safety Info_literacy  Compulsory_years  immig_pct  corr_base_sens  R2_res  R2_part
AUT         Compulsory   Non-compulsory           Explicit           Explicit                      Yes     Explicit      Explicit                 9       27.0           -0.71   0.031    0.235
DEU           Optional             None           Explicit           Explicit                      Yes     Explicit      Explicit                 9       26.1            0.35  -0.176    0.285
DNK         Not taught             None           Explicit           Explicit                      Yes     Explicit      Explicit                10       10.5           -0.56  -0.021    0.279
FIN           Optional             None           Explicit           Explicit                       No     Explicit      Explicit                11        6.2           -0.90   0.012    0.408
FRA         Not taught     School-level 